# E-commerce A/B Testing with Revenue Forecasting

## Notebook 3: Feature Engineering

This notebook creates the modeling features for the A/B testing and revenue forecasting workflows. The goal is to transform cleaned source data into structured feature tables that support forecasting, uplift translation, and scenario analysis.

## Goals

1. Create A/B experiment summary features
2. Build a daily revenue forecasting table
3. Create calendar, lag, and rolling features
4. Add oil and transaction features
5. Save feature-ready datasets for modeling

In [1]:
# Import libraries

from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
# Set display options and project paths

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 100)

PROJECT_ROOT = Path.cwd().parent
PREPROCESSED_DIR = PROJECT_ROOT / "data" / "02-preprocessed"
FEATURES_DIR = PROJECT_ROOT / "data" / "03-features"

FEATURES_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Preprocessed dir exists:", PREPROCESSED_DIR.exists())
print("Features dir exists:", FEATURES_DIR.exists())

Project root: c:\Users\tevin\OneDrive\Desktop\General-Machine-Learning\revenue_forecasting
Preprocessed dir exists: True
Features dir exists: True


In [3]:
# Load input datasets

ab_df = pd.read_csv(PREPROCESSED_DIR / "ab_test_clean.csv", parse_dates=["date"])
train_df = pd.read_csv(PREPROCESSED_DIR / "store_sales_train_clean.csv", parse_dates=["date"])
transactions_df = pd.read_csv(PREPROCESSED_DIR / "store_sales_transactions_clean.csv", parse_dates=["date"])
oil_df = pd.read_csv(PREPROCESSED_DIR / "store_sales_oil_clean.csv", parse_dates=["date"])
stores_df = pd.read_csv(PREPROCESSED_DIR / "store_sales_stores_clean.csv")
holidays_df = pd.read_csv(PREPROCESSED_DIR / "store_sales_holidays_clean.csv", parse_dates=["date"])

daily_sales = pd.read_csv(PREPROCESSED_DIR / "eda_daily_sales.csv", parse_dates=["date"])
ab_daily = pd.read_csv(PREPROCESSED_DIR / "eda_ab_daily.csv", parse_dates=["date"])
daily_transactions = pd.read_csv(PREPROCESSED_DIR / "eda_daily_transactions.csv", parse_dates=["date"])
sales_transactions_df = pd.read_csv(PREPROCESSED_DIR / "eda_sales_transactions.csv", parse_dates=["date"])
sales_oil_df = pd.read_csv(PREPROCESSED_DIR / "eda_sales_oil.csv", parse_dates=["date"])

print("ab_df:", ab_df.shape)
print("daily_sales:", daily_sales.shape)
print("daily_transactions:", daily_transactions.shape)
print("sales_oil_df:", sales_oil_df.shape)

ab_df: (60, 11)
daily_sales: (1684, 7)
daily_transactions: (1682, 2)
sales_oil_df: (1684, 8)


## A/B Experiment Feature Table

This section creates a compact A/B experiment summary table with variant-level rates and uplift metrics.

In [4]:
# Build A/B variant summary feature table

ab_variant_features = ab_df.groupby("variant", as_index=False).agg(
    total_spend=("spend_usd", "sum"),
    total_impressions=("impressions", "sum"),
    total_reach=("reach", "sum"),
    total_clicks=("website_clicks", "sum"),
    total_searches=("searches", "sum"),
    total_view_content=("view_content", "sum"),
    total_add_to_cart=("add_to_cart", "sum"),
    total_purchases=("purchase", "sum")
)

ab_variant_features["ctr"] = ab_variant_features["total_clicks"] / ab_variant_features["total_impressions"]
ab_variant_features["purchase_rate_from_click"] = ab_variant_features["total_purchases"] / ab_variant_features["total_clicks"]
ab_variant_features["purchase_rate_from_impression"] = ab_variant_features["total_purchases"] / ab_variant_features["total_impressions"]
ab_variant_features["cost_per_purchase"] = ab_variant_features["total_spend"] / ab_variant_features["total_purchases"]

ab_variant_features

,variant,total_spend,total_impressions,total_reach,total_clicks,total_searches,total_view_content,total_add_to_cart,total_purchases,ctr,purchase_rate_from_click,purchase_rate_from_impression,cost_per_purchase
0,control,68653,3177233.0,2576503.0,154303.0,64418.0,56370.0,37700.0,15161.0,0.048565,0.098255,0.004772,4.528263
1,test,76892,2237544.0,1604747.0,180970.0,72569.0,55740.0,26446.0,15637.0,0.080879,0.086407,0.006988,4.917312


In [5]:
# Create uplift summary from control to test

control_metrics = ab_variant_features[ab_variant_features["variant"] == "control"].iloc[0]
test_metrics = ab_variant_features[ab_variant_features["variant"] == "test"].iloc[0]

ab_uplift_summary = pd.DataFrame({
    "metric": [
        "ctr",
        "purchase_rate_from_click",
        "purchase_rate_from_impression",
        "cost_per_purchase"
    ],
    "control_value": [
        control_metrics["ctr"],
        control_metrics["purchase_rate_from_click"],
        control_metrics["purchase_rate_from_impression"],
        control_metrics["cost_per_purchase"]
    ],
    "test_value": [
        test_metrics["ctr"],
        test_metrics["purchase_rate_from_click"],
        test_metrics["purchase_rate_from_impression"],
        test_metrics["cost_per_purchase"]
    ]
})

ab_uplift_summary["absolute_diff"] = ab_uplift_summary["test_value"] - ab_uplift_summary["control_value"]
ab_uplift_summary["relative_uplift_pct"] = (
    ab_uplift_summary["absolute_diff"] / ab_uplift_summary["control_value"]
) * 100

ab_uplift_summary

,metric,control_value,test_value,absolute_diff,relative_uplift_pct
0,ctr,0.048565,0.080879,0.032314,66.536601
1,purchase_rate_from_click,0.098255,0.086407,-0.011848,-12.058601
2,purchase_rate_from_impression,0.004772,0.006988,0.002217,46.454617
3,cost_per_purchase,4.528263,4.917312,0.389048,8.591554


## Daily Forecasting Base Table

This section builds the main daily forecasting table that will later receive calendar, lag, rolling, and external features.

In [6]:
# Build base daily forecasting table

forecast_df = daily_sales.merge(
    daily_transactions,
    on="date",
    how="left"
).merge(
    oil_df[["date", "dcoilwtico"]],
    on="date",
    how="left"
)

forecast_df = forecast_df.sort_values("date").reset_index(drop=True)

forecast_df.head()

,date,total_sales,total_onpromotion,year,month,day_of_week,day_of_week_num,total_transactions,dcoilwtico
0,2013-01-01,2511.618999,0,2013,1,Tuesday,1,770.0,NaN
1,2013-01-02,496092.417944,0,2013,1,Wednesday,2,93215.0,93.14
2,2013-01-03,361461.231124,0,2013,1,Thursday,3,78504.0,92.97
3,2013-01-04,354459.677093,0,2013,1,Friday,4,78494.0,93.12
4,2013-01-05,477350.121229,0,2013,1,Saturday,5,93573.0,NaN


## Calendar Features

This section creates basic calendar features that capture recurring weekly, monthly, and quarterly patterns.

In [7]:
# Create calendar features

forecast_df["month"] = forecast_df["date"].dt.month
forecast_df["day_of_week_num"] = forecast_df["date"].dt.dayofweek
forecast_df["week_of_year"] = forecast_df["date"].dt.isocalendar().week.astype(int)
forecast_df["is_month_start"] = forecast_df["date"].dt.is_month_start.astype(int)
forecast_df["is_month_end"] = forecast_df["date"].dt.is_month_end.astype(int)

forecast_df.head()

,date,total_sales,total_onpromotion,year,month,day_of_week,day_of_week_num,total_transactions,dcoilwtico,week_of_year,is_month_start,is_month_end
0,2013-01-01,2511.618999,0,2013,1,Tuesday,1,770.0,NaN,1,1,0
1,2013-01-02,496092.417944,0,2013,1,Wednesday,2,93215.0,93.14,1,0,0
2,2013-01-03,361461.231124,0,2013,1,Thursday,3,78504.0,92.97,1,0,0
3,2013-01-04,354459.677093,0,2013,1,Friday,4,78494.0,93.12,1,0,0
4,2013-01-05,477350.121229,0,2013,1,Saturday,5,93573.0,NaN,1,0,0


## Cyclical Time Features

This section encodes cyclical calendar patterns using sine and cosine transformations.

In [8]:
# Create cyclical time features

forecast_df["dow_sin"] = np.sin(2 * np.pi * forecast_df["day_of_week"] / 7)
forecast_df["dow_cos"] = np.cos(2 * np.pi * forecast_df["day_of_week"] / 7)

forecast_df["month_sin"] = np.sin(2 * np.pi * forecast_df["month"] / 12)
forecast_df["month_cos"] = np.cos(2 * np.pi * forecast_df["month"] / 12)

forecast_df.head()

TypeError: can't multiply sequence by non-int of type 'float'

## Lag Features

This section creates lagged sales and promotion features so the models can learn from past demand patterns.

In [ ]:
# Create lag features for sales

forecast_df["sales_lag_1"] = forecast_df["total_sales"].shift(1)
forecast_df["sales_lag_7"] = forecast_df["total_sales"].shift(7)
forecast_df["sales_lag_14"] = forecast_df["total_sales"].shift(14)
forecast_df["sales_lag_28"] = forecast_df["total_sales"].shift(28)

forecast_df["promo_lag_1"] = forecast_df["total_onpromotion"].shift(1)
forecast_df["promo_lag_7"] = forecast_df["total_onpromotion"].shift(7)

forecast_df.head(15)

## Rolling Features

This section creates rolling means and rolling volatility features for sales and promotions.

In [ ]:
# Create rolling features

forecast_df["sales_roll_mean_7"] = forecast_df["total_sales"].shift(1).rolling(7).mean()
forecast_df["sales_roll_mean_14"] = forecast_df["total_sales"].shift(1).rolling(14).mean()
forecast_df["sales_roll_mean_28"] = forecast_df["total_sales"].shift(1).rolling(28).mean()

forecast_df["sales_roll_std_7"] = forecast_df["total_sales"].shift(1).rolling(7).std()
forecast_df["sales_roll_std_14"] = forecast_df["total_sales"].shift(1).rolling(14).std()

forecast_df["promo_roll_mean_7"] = forecast_df["total_onpromotion"].shift(1).rolling(7).mean()
forecast_df["promo_roll_mean_14"] = forecast_df["total_onpromotion"].shift(1).rolling(14).mean()

forecast_df.head(20)

## Growth and Change Features

This section creates percentage-change features for sales, transactions, and oil prices.

In [ ]:
# Create growth and change features

forecast_df["sales_pct_change_1"] = forecast_df["total_sales"].pct_change(1)
forecast_df["sales_pct_change_7"] = forecast_df["total_sales"].pct_change(7)

forecast_df["transactions_pct_change_1"] = forecast_df["total_transactions"].pct_change(1)
forecast_df["transactions_pct_change_7"] = forecast_df["total_transactions"].pct_change(7)

forecast_df["oil_pct_change_1"] = forecast_df["dcoilwtico"].pct_change(1)
forecast_df["oil_pct_change_7"] = forecast_df["dcoilwtico"].pct_change(7)

forecast_df.head(20)

## Oil Features

This section fills oil-price gaps and creates smoothed oil features for forecasting.

In [ ]:
# Fill and smooth oil features

forecast_df["dcoilwtico_filled"] = forecast_df["dcoilwtico"].ffill().bfill()
forecast_df["oil_roll_mean_7"] = forecast_df["dcoilwtico_filled"].rolling(7).mean()
forecast_df["oil_roll_mean_14"] = forecast_df["dcoilwtico_filled"].rolling(14).mean()

forecast_df.head(20)

## Holiday Features

This section creates a basic holiday flag from the holidays table.

In [9]:
# Create holiday features

holiday_flags = holidays_df.copy()
holiday_flags["is_holiday"] = 1

holiday_flags = holiday_flags.groupby("date", as_index=False).agg(
    is_holiday=("is_holiday", "max")
)

forecast_df = forecast_df.merge(
    holiday_flags,
    on="date",
    how="left"
)

forecast_df["is_holiday"] = forecast_df["is_holiday"].fillna(0).astype(int)

forecast_df.head()

,date,total_sales,total_onpromotion,year,month,day_of_week,day_of_week_num,total_transactions,dcoilwtico,week_of_year,is_month_start,is_month_end,is_holiday
0,2013-01-01,2511.618999,0,2013,1,Tuesday,1,770.0,NaN,1,1,0,1
1,2013-01-02,496092.417944,0,2013,1,Wednesday,2,93215.0,93.14,1,0,0,0
2,2013-01-03,361461.231124,0,2013,1,Thursday,3,78504.0,92.97,1,0,0,0
3,2013-01-04,354459.677093,0,2013,1,Friday,4,78494.0,93.12,1,0,0,0
4,2013-01-05,477350.121229,0,2013,1,Saturday,5,93573.0,NaN,1,0,0,1


## Scenario Multipliers

This section translates A/B uplift into simple scenario multipliers for later revenue forecasting scenarios.

In [10]:
# Create scenario multipliers from A/B uplift

purchase_uplift_pct = ab_uplift_summary.loc[
    ab_uplift_summary["metric"] == "purchase_rate_from_click",
    "relative_uplift_pct"
].iloc[0]

scenario_multipliers = pd.DataFrame({
    "scenario": ["conservative", "base", "optimistic"],
    "uplift_pct": [
        purchase_uplift_pct * 0.50,
        purchase_uplift_pct,
        purchase_uplift_pct * 1.25
    ]
})

scenario_multipliers["multiplier"] = 1 + (scenario_multipliers["uplift_pct"] / 100)

scenario_multipliers

,scenario,uplift_pct,multiplier
0,conservative,-6.029300,0.939707
1,base,-12.058601,0.879414
2,optimistic,-15.073251,0.849267


## Final Modeling Table

This section filters the forecasting table into a clean model-ready dataset after lag and rolling features are created.

In [14]:
# Check current columns in forecast_df

forecast_df.columns.tolist()

['date',
 'total_sales',
 'total_onpromotion',
 'year',
 'month',
 'day_of_week',
 'day_of_week_num',
 'total_transactions',
 'dcoilwtico',
 'week_of_year',
 'is_month_start',
 'is_month_end',
 'is_holiday']

In [15]:
# Create lag features

forecast_df["sales_lag_1"] = forecast_df["total_sales"].shift(1)
forecast_df["sales_lag_7"] = forecast_df["total_sales"].shift(7)
forecast_df["sales_lag_14"] = forecast_df["total_sales"].shift(14)
forecast_df["sales_lag_28"] = forecast_df["total_sales"].shift(28)

forecast_df["promo_lag_1"] = forecast_df["total_onpromotion"].shift(1)
forecast_df["promo_lag_7"] = forecast_df["total_onpromotion"].shift(7)

forecast_df.head(15)

,date,total_sales,total_onpromotion,year,month,day_of_week,day_of_week_num,total_transactions,dcoilwtico,week_of_year,is_month_start,is_month_end,is_holiday,sales_lag_1,sales_lag_7,sales_lag_14,sales_lag_28,promo_lag_1,promo_lag_7
0,2013-01-01,2511.618999,0,2013,1,Tuesday,1,770.0,NaN,1,1,0,1,NaN,NaN,NaN,NaN,NaN,NaN
1,2013-01-02,496092.417944,0,2013,1,Wednesday,2,93215.0,93.14,1,0,0,0,2511.618999,NaN,NaN,NaN,0.0,NaN
2,2013-01-03,361461.231124,0,2013,1,Thursday,3,78504.0,92.97,1,0,0,0,496092.417944,NaN,NaN,NaN,0.0,NaN
3,2013-01-04,354459.677093,0,2013,1,Friday,4,78494.0,93.12,1,0,0,0,361461.231124,NaN,NaN,NaN,0.0,NaN
4,2013-01-05,477350.121229,0,2013,1,Saturday,5,93573.0,NaN,1,0,0,1,354459.677093,NaN,NaN,NaN,0.0,NaN
5,2013-01-06,519695.401088,0,2013,1,Sunday,6,90464.0,NaN,1,0,0,0,477350.121229,NaN,NaN,NaN,0.0,NaN
6,2013-01-07,336122.801066,0,2013,1,Monday,0,75597.0,93.20,2,0,0,0,519695.401088,NaN,NaN,NaN,0.0,NaN
7,2013-01-08,318347.777981,0,2013,1,Tuesday,1,72325.0,93.21,2,0,0,0,336122.801066,2511.618999,NaN,NaN,0.0,0.0
8,2013-01-09,302530.809018,0,2013,1,Wednesday,2,71971.0,93.08,2,0,0,0,318347.777981,496092.417944,NaN,NaN,0.0,0.0
9,2013-01-10,258982.003049,0,2013,1,Thursday,3,66383.0,93.81,2,0,0,0,302530.809018,361461.231124,NaN,NaN,0.0,0.0


In [16]:
# Create rolling features

forecast_df["sales_roll_mean_7"] = forecast_df["total_sales"].shift(1).rolling(7).mean()
forecast_df["sales_roll_mean_14"] = forecast_df["total_sales"].shift(1).rolling(14).mean()
forecast_df["sales_roll_mean_28"] = forecast_df["total_sales"].shift(1).rolling(28).mean()

forecast_df["sales_roll_std_7"] = forecast_df["total_sales"].shift(1).rolling(7).std()
forecast_df["sales_roll_std_14"] = forecast_df["total_sales"].shift(1).rolling(14).std()

forecast_df["promo_roll_mean_7"] = forecast_df["total_onpromotion"].shift(1).rolling(7).mean()
forecast_df["promo_roll_mean_14"] = forecast_df["total_onpromotion"].shift(1).rolling(14).mean()

forecast_df.head(20)

,date,total_sales,total_onpromotion,year,month,day_of_week,day_of_week_num,total_transactions,dcoilwtico,week_of_year,is_month_start,is_month_end,is_holiday,sales_lag_1,sales_lag_7,sales_lag_14,sales_lag_28,promo_lag_1,promo_lag_7,sales_roll_mean_7,sales_roll_mean_14,sales_roll_mean_28,sales_roll_std_7,sales_roll_std_14,promo_roll_mean_7,promo_roll_mean_14
0,2013-01-01,2511.618999,0,2013,1,Tuesday,1,770.0,NaN,1,1,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2013-01-02,496092.417944,0,2013,1,Wednesday,2,93215.0,93.14,1,0,0,0,2511.618999,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2013-01-03,361461.231124,0,2013,1,Thursday,3,78504.0,92.97,1,0,0,0,496092.417944,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2013-01-04,354459.677093,0,2013,1,Friday,4,78494.0,93.12,1,0,0,0,361461.231124,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2013-01-05,477350.121229,0,2013,1,Saturday,5,93573.0,NaN,1,0,0,1,354459.677093,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2013-01-06,519695.401088,0,2013,1,Sunday,6,90464.0,NaN,1,0,0,0,477350.121229,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2013-01-07,336122.801066,0,2013,1,Monday,0,75597.0,93.20,2,0,0,0,519695.401088,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2013-01-08,318347.777981,0,2013,1,Tuesday,1,72325.0,93.21,2,0,0,0,336122.801066,2511.618999,NaN,NaN,0.0,0.0,363956.181220,NaN,NaN,176108.939896,NaN,0.0,NaN
8,2013-01-09,302530.809018,0,2013,1,Wednesday,2,71971.0,93.08,2,0,0,0,318347.777981,496092.417944,NaN,NaN,0.0,0.0,409075.632504,NaN,NaN,84925.215107,NaN,0.0,NaN
9,2013-01-10,258982.003049,0,2013,1,Thursday,3,66383.0,93.81,2,0,0,0,302530.809018,361461.231124,NaN,NaN,0.0,0.0,381423.974086,NaN,NaN,83367.991396,NaN,0.0,NaN


In [17]:
# Create oil features

forecast_df["dcoilwtico_filled"] = forecast_df["dcoilwtico"].ffill().bfill()
forecast_df["oil_roll_mean_7"] = forecast_df["dcoilwtico_filled"].rolling(7).mean()
forecast_df["oil_roll_mean_14"] = forecast_df["dcoilwtico_filled"].rolling(14).mean()

forecast_df.head(20)

,date,total_sales,total_onpromotion,year,month,day_of_week,day_of_week_num,total_transactions,dcoilwtico,week_of_year,is_month_start,is_month_end,is_holiday,sales_lag_1,sales_lag_7,sales_lag_14,sales_lag_28,promo_lag_1,promo_lag_7,sales_roll_mean_7,sales_roll_mean_14,sales_roll_mean_28,sales_roll_std_7,sales_roll_std_14,promo_roll_mean_7,promo_roll_mean_14,dcoilwtico_filled,oil_roll_mean_7,oil_roll_mean_14
0,2013-01-01,2511.618999,0,2013,1,Tuesday,1,770.0,NaN,1,1,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,93.14,NaN,NaN
1,2013-01-02,496092.417944,0,2013,1,Wednesday,2,93215.0,93.14,1,0,0,0,2511.618999,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,93.14,NaN,NaN
2,2013-01-03,361461.231124,0,2013,1,Thursday,3,78504.0,92.97,1,0,0,0,496092.417944,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,92.97,NaN,NaN
3,2013-01-04,354459.677093,0,2013,1,Friday,4,78494.0,93.12,1,0,0,0,361461.231124,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,93.12,NaN,NaN
4,2013-01-05,477350.121229,0,2013,1,Saturday,5,93573.0,NaN,1,0,0,1,354459.677093,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,93.12,NaN,NaN
5,2013-01-06,519695.401088,0,2013,1,Sunday,6,90464.0,NaN,1,0,0,0,477350.121229,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,93.12,NaN,NaN
6,2013-01-07,336122.801066,0,2013,1,Monday,0,75597.0,93.20,2,0,0,0,519695.401088,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,93.20,93.115714,NaN
7,2013-01-08,318347.777981,0,2013,1,Tuesday,1,72325.0,93.21,2,0,0,0,336122.801066,2511.618999,NaN,NaN,0.0,0.0,363956.181220,NaN,NaN,176108.939896,NaN,0.0,NaN,93.21,93.125714,NaN
8,2013-01-09,302530.809018,0,2013,1,Wednesday,2,71971.0,93.08,2,0,0,0,318347.777981,496092.417944,NaN,NaN,0.0,0.0,409075.632504,NaN,NaN,84925.215107,NaN,0.0,NaN,93.08,93.117143,NaN
9,2013-01-10,258982.003049,0,2013,1,Thursday,3,66383.0,93.81,2,0,0,0,302530.809018,361461.231124,NaN,NaN,0.0,0.0,381423.974086,NaN,NaN,83367.991396,NaN,0.0,NaN,93.81,93.237143,NaN


In [18]:
# Create final model-ready forecasting table

model_df = forecast_df.copy()

required_feature_cols = [
    "sales_lag_1",
    "sales_lag_7",
    "sales_lag_14",
    "sales_lag_28",
    "sales_roll_mean_7",
    "sales_roll_mean_14",
    "sales_roll_mean_28",
    "sales_roll_std_7",
    "sales_roll_std_14",
    "promo_lag_1",
    "promo_lag_7",
    "promo_roll_mean_7",
    "promo_roll_mean_14",
    "dcoilwtico_filled",
    "oil_roll_mean_7",
    "oil_roll_mean_14",
    "total_transactions"
]

model_df = model_df.dropna(subset=required_feature_cols).reset_index(drop=True)

print("forecast_df shape:", forecast_df.shape)
print("model_df shape:", model_df.shape)

model_df.head()

forecast_df shape: (1684, 29)
model_df shape: (1654, 29)


,date,total_sales,total_onpromotion,year,month,day_of_week,day_of_week_num,total_transactions,dcoilwtico,week_of_year,is_month_start,is_month_end,is_holiday,sales_lag_1,sales_lag_7,sales_lag_14,sales_lag_28,promo_lag_1,promo_lag_7,sales_roll_mean_7,sales_roll_mean_14,sales_roll_mean_28,sales_roll_std_7,sales_roll_std_14,promo_roll_mean_7,promo_roll_mean_14,dcoilwtico_filled,oil_roll_mean_7,oil_roll_mean_14
0,2013-01-29,264488.818077,0,2013,1,Tuesday,1,68435.0,97.62,5,0,0,0,285460.169953,296214.728983,299129.549954,2511.618999,0.0,0.0,320916.224872,330877.422002,339672.163349,71044.389805,70726.570278,0.0,0.0,97.62,95.632857,95.552143
1,2013-01-30,281061.127052,0,2013,1,Wednesday,2,70888.0,97.98,5,0,0,0,264488.818077,283258.453032,318347.913946,496092.417944,0.0,0.0,316383.951885,328403.084010,349028.491888,73839.837696,72506.247548,0.0,0.0,97.98,96.050000,95.816429
2,2013-01-31,271254.217996,0,2013,1,Thursday,3,70268.0,97.65,5,0,1,0,281061.127052,247245.690995,267498.515975,361461.231124,0.0,0.0,316070.048174,325739.742089,341348.802928,74008.606838,73580.868185,0.0,0.0,97.65,96.378571,95.970714
3,2013-02-01,369402.055266,0,2013,2,Friday,4,78302.0,97.46,5,1,0,0,271254.217996,290022.771930,296130.850028,354459.677093,0.0,0.0,319499.837745,326008.006519,338127.123887,70773.071168,73358.706951,0.0,0.0,97.46,96.708571,96.102857
4,2013-02-02,518887.462705,0,2013,2,Saturday,5,97347.0,NaN,5,0,0,0,369402.055266,413799.767975,432459.852021,477350.121229,0.0,0.0,330839.735365,331241.664036,338660.780251,71617.207631,73676.225736,0.0,0.0,97.46,97.038571,96.235000


## Feature Table Validation

This section checks the final engineered tables for missing values and confirms they are ready for modeling.

In [12]:
# Check missing values in final model table

missing_summary = model_df.isna().sum().reset_index()
missing_summary.columns = ["column", "missing_count"]
missing_summary = missing_summary[missing_summary["missing_count"] > 0].sort_values("missing_count", ascending=False)

missing_summary

,column,missing_count
8,dcoilwtico,521
7,total_transactions,2


## Save Feature Tables

This section writes the engineered experiment and forecasting tables to the features directory.

In [20]:
# Save engineered feature tables

ab_variant_features.to_csv(FEATURES_DIR / "ab_variant_features.csv", index=False)
ab_uplift_summary.to_csv(FEATURES_DIR / "ab_uplift_summary.csv", index=False)
scenario_multipliers.to_csv(FEATURES_DIR / "scenario_multipliers.csv", index=False)

forecast_df.to_csv(FEATURES_DIR / "forecast_daily_feature_base.csv", index=False)
model_df.to_csv(FEATURES_DIR / "forecast_modeling_table.csv", index=False)

print("Feature tables saved successfully.")

Feature tables saved successfully.
